# Non-linear effects from infectious disease models
This notebook will explore how non-linear effects of public health interventions against infectious diseases can be reflected in mechanistic models such as the one introduced in the previous notebook (`01-sir-intro.ipynb`).

We'll start off with the similar package installation code (just a couple more commonly used Python packages), which you can again ignore.

In [ ]:
# !pip install git+https://github.com/monash-emu/summer3wip.git

In [ ]:
import numpy as np
from plotly import graph_objects as go
import pandas as pd
pd.options.plotting.backend = "plotly"

from summer3.graph import defer, Parameter, CompartmentValues
from summer3.epi import Stratification, CompartmentMap, \
    ManagedArray, CategoryData, TransitionFlow, CompartmentalEpiModel

## Building the base model
Next, we'll build a model that is very similar to the one in the last notebook, with almost identical code, except that we won't incorporate the process of infection into the model yet.

In [ ]:
population = 1e6  
seed = 1.0
start_time = 0
end_time = 50
model_comps = ["susceptible", "infectious", "recovered"]
infect_comps = ["infectious"]
disease_state = Stratification("disease_state", model_comps)
humans = CompartmentMap.new(disease_state)
epi_model = CompartmentalEpiModel(humans, range(start_time, end_time))
start_pop = [population - seed, seed, 0.0]
epi_model.set_initial_population(CategoryData(disease_state.categories(), np.array(start_pop)))

def iprocess(compartment_values: ManagedArray, contact_rate: float, infectious_compartments: tuple):
    n_infectious = compartment_values.query(infectious_compartments).sum()
    n_population = compartment_values.sum()
    infectious_prevalence = n_infectious / n_population
    return contact_rate * infectious_prevalence

recovery = TransitionFlow("recovery", disease_state["infectious"], disease_state["recovered"], Parameter("recovery_rate", 0.0))
epi_model.add_flow(recovery)
base_parameters = {"effective_contact_rate": 1.2, "recovery_rate": 0.2}

## Adding an intervention
Let's consider the effect of a public health intervention on the dynamics we observed with the base model in the previous noteboook. To capture this, we'll imagine that the government has mandated some restrictions that reduce the rate at which people come into contact with one-another, i.e. a "lockdown". We'll implement this in the model by assuming some strength of the effect of lockdown on transmission, so that a stronger lockdown reduces transmission further. The following cell achieves this in code by scaling back the rate at which people get infected (for a certain prevalence of infection) according to the effect of the intervention.

![](../images/sir_lockdown.svg)

In [ ]:
# Scale back the rate of contact according to the effect of the lockdown
lockdown_adjustment = 1.0 - Parameter("lockdown_effect", 0.0)
contact_rate = Parameter("effective_contact_rate", 0.0) * lockdown_adjustment

# Then add the flow to the model as we did previously
force_of_infection = defer(iprocess)(CompartmentValues, contact_rate, disease_state["infectious"])
infection = TransitionFlow("infection", disease_state["susceptible"], disease_state["infectious"], force_of_infection)
epi_model.add_flow(infection)

### Run the model with various lockdown effects
Now let's run the model with various lockdown effect strength parameters and see how it plays out in the infection dynamics.

In [ ]:
# Effects vary from zero to one in 10% increments
lockdown_effects = np.arange(0.0, 1.1, 0.1)

# Prepare for the outputs
outputs = pd.DataFrame(columns=lockdown_effects)

# Cycle through the effects
for effect in lockdown_effects:

    # Add the lockdown effect to the base parameter set and run
    results = epi_model.run(base_parameters | {"lockdown_effect": effect})

    # Collate the results
    outputs[effect] = results["compartments"].to_pandas_df()["infectious"]

### Plotting the epi-curve
First let's consider what that looks like in terms of the epi-curve of cases over time.

In [ ]:
fig = go.Figure()
legend_format = {"title": "lockdown effect"}
xaxis_format = {"title": "days"}
for c in outputs.columns:
    colour = f"rgba({c}, 0, {1.0 - c}, 1)"
    fig.add_trace(go.Scatter(x=outputs.index, y=outputs[c], mode="lines", name=round(c, 1), line={"color": colour}))
fig.update_layout(title="effect of lockdown severity on the epidemic", legend=legend_format, xaxis=xaxis_format, yaxis={"title": "infection prevalence"})

### Interpretation
What do you notice here?

A few things should stand out. Obviously, the epidemic has changed, and the change we get isn't obviously predictable from the severity of the restrictions we implemented. Not only does the size of the epidemic change, but so does it's shape. If you toggle off the lines for lockdown effects from zero through to 0.8, you'll also see that the epi-curve just goes down for 0.9 and 1.0 severity. What might this mean in terms of peak hospital demand for this epidemic? What might we call this phenomenon?

To get a better sense of the total size of the epidemic, let's have a look at how that looks in terms of cumulative cases rather than cases per day. Are we actually having any effect on the total size of the epidemic by the time it has completed?

In [ ]:
# Calculate the cumulative numbers from the daily
cum_outputs = outputs.cumsum()

# Plot with colour scaling from blue to red
cum_fig = go.Figure()
for c in cum_outputs.columns:
    colour = f"rgba({c}, 0, {1.0 - c}, 1)"
    cum_fig.add_trace(go.Scatter(x=cum_outputs.index, y=cum_outputs[c], mode="lines", name=round(c, 1), line={"color": colour}))
cum_fig.update_layout(title="effect of lockdown severity on cumulative cases", legend=legend_format, xaxis=xaxis_format, yaxis={"title": "cumulative cases"})

To get an even better sense of these effects, rather than looking at the cumulative cases over time, the non-linear effect might be clearer if we look at the size of the epidemic by 50 days for various lockdown strengths.

In [ ]:
# Get the cumulative cases by the last day for each lockdown severity value
cum_cases_day50 = cum_outputs.iloc[-1]

# Plot
nonlinear_fig = go.Figure()
nonlinear_fig.add_trace(go.Scatter(x=cum_outputs.columns, y=cum_cases_day50))
nonlinear_fig.update_layout(title="cumulative cases at 50 days by lockdown effect", xaxis={"title": "lockdown effect"}, yaxis={"title": "cumulative cases"})

Now we can more clearly see the threshold effects of this intervention as it increases in strength.